# Data Augmentation

![Data augmentation example](https://i0.wp.com/ubiai.tools/wp-content/uploads/2023/11/0_LNtz0G4cngapDH41.png?fit=960%2C540&ssl=1)


## Key idea

Data augmentation means creating *slightly changed* versions of your training images while keeping the label the same. 

> The goal is to help a model learn what matters (the object/shape) and ignore small, common changes (like viewpoint or lighting).

This usually improves performance on new images because the model has “seen” more variety during training.

### Why it helps
- **Real-world images vary**: a camera can be tilted, an object can appear on the left or right, or the room can be brighter or darker.
- **If training images are too “clean” and repetitive**, a model can memorize tiny details (background, exact position) instead of learning the general pattern.
- Augmentation teaches the model that these small changes should not change the answer.



### Common augmentations (and what they simulate)

> A good augmentation copies real-world variation your camera could produce, without changing what a human would call the image.

- **Flip**: Mirror the image left↔right (sometimes up↕down). This simulates the same object appearing from the opposite side or the camera being held differently. It’s often safe for animals and everyday objects, but not always safe for things where direction matters (see rule below).

- **Crop / zoom**: Take a smaller window (crop) or scale in/out (zoom). This simulates the object being closer/farther, partially out of frame, or not perfectly centered. It also reduces over-reliance on the background because the model can’t count on seeing the full scene every time.

- **Rotate**: Turn the image by a small angle. This simulates a slightly tilted camera or a natural change in viewpoint. Small rotations are common in real photos; very large rotations can create unrealistic examples for many tasks.

- **Contrast / brightness**: Change how light/dark the image is (brightness) and how strong the differences are between light and dark areas (contrast). This simulates different lighting conditions: indoor vs outdoor, shadows, cloudy vs sunny, or a camera with different exposure settings.

- **Hue**: Shift the “color tone” (e.g., a slight move toward warmer or cooler colors). This simulates changes in white balance, different light sources (sunlight vs fluorescent), or different camera sensors. Use small hue shifts so colors still look realistic.

### Important rule: keep labels correct

> Always ask: “After this change, would a person still give the same label?” If the answer is not clearly “yes,” reduce that augmentation or remove it.

Augmentations must not change the correct answer (label). If the transformation changes the meaning, you are training the model on wrong examples, which usually hurts performance.

Examples where you need to be careful:
- **Text / digits**: Flipping can make letters look like different letters, and rotation can make “6” look like “9” in some styles.
- **Directional signs / arrows**: Left-right flip can change “turn left” into “turn right.”
- **Medical or scientific images**: Orientation and color may carry meaning; random hue/rotation could be misleading depending on the task.



## Data augmentation tools you’ll use in practice

> Use `tf.image` when you want *full control* and very clear “input → operation → output” code.

### TensorFlow: `tf.image` (function-based)
`tf.image` is a collection of image operations you can call directly, which makes it great for “show one transform at a time” demos in a notebook and for use inside a `tf.data` pipeline. 
It includes flips and common color changes such as brightness, contrast, and hue (both deterministic “adjust\_*” and random “random\_*” versions).

Common examples:  
- `tf.image.flip_left_right(image)` flips an image horizontally. 
- `tf.image.random_flip_left_right(image)` does the same, but only sometimes (randomly). 
- `tf.image.adjust_hue(image, delta)` changes hue by converting RGB→HSV, rotating the hue channel, then converting back to RGB; `delta` must be in `[-1, 1]`.



### Keras: preprocessing augmentation layers (model-based)

> Use Keras layers when you want a clean “augmentation block” you can reuse across models with minimal code.

Keras provides built-in augmentation layers like `RandomFlip` and `RandomRotation` that you can stack together (often inside a `tf.keras.Sequential`) and plug into your model.
TensorFlow’s own tutorial shows this pattern and demonstrates putting preprocessing layers directly into a `tf.keras.Sequential` model. 
These layers are documented as part of Keras “image augmentation” preprocessing layers. 


### OpenCV: `cv2` (computer-vision toolbox)

> Use OpenCV when your pipeline is already “image-processing heavy,” or you need fine control over geometry and borders during transforms. 

OpenCV is widely used for classic image processing and gives you direct tools for geometric transforms such as rotation/translation using `cv2.getRotationMatrix2D(...)` and applying it with `cv2.warpAffine(...)`. 

This is useful when you want to augment images *before* training (e.g., to save augmented files) or when you’re already doing other OpenCV-style preprocessing. 



## Practical part: apply augmentations and compare visually

> Before we automate augmentation inside a training pipeline, we want to clearly see what each transformation does and when it might help (or hurt) our labels.

In the next cells, we will:

1. Load **one image**.
2. Apply a set of transformations **one by one** using the TensorFlow image functions (`tf.image.*`). 
3. After each transformation, we will plot the **original** image and the **augmented** image **side by side** so it’s easy to see what changed. 

We will cover these transformations:

- Flip (left ↔ right)
- Crop
- Rotate
- Contrast
- Brightness
- Hue

In [13]:
import numpy as np
import matplotlib.pyplot as plt

def _to_displayable(img):
    """Convert tf.Tensor / numpy / PIL-like to a numpy array Matplotlib can show."""
    if hasattr(img, "numpy"):  # tf.Tensor (eager)
        img = img.numpy()
    img = np.asarray(img)

    # If float image is in [0, 255], bring it to [0, 1] for nicer display (optional).
    if img.dtype.kind == "f" and img.max() > 1.0:
        img = img / 255.0

    # If there's a batch dimension (1, H, W, C), remove it.
    if img.ndim == 4 and img.shape[0] == 1:
        img = img[0]

    return img

def show_side_by_side(original, augmented, titles=("Original", "Augmented"), figsize=(10, 4)):
    """Display two images side by side."""
    original = _to_displayable(original)
    augmented = _to_displayable(augmented)

    fig, axes = plt.subplots(1, 2, figsize=figsize)  # 1 row, 2 columns 
    for ax, img, title in zip(axes, [original, augmented], titles):
        ax.imshow(img)  # show image array 
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


In [15]:
import tensorflow as tf

# Load the image (PNG) as a tensor: shape (H, W, 3), dtype uint8
image_bytes = tf.io.read_file("homer.png")
image = tf.io.decode_png(image_bytes, channels=3) 

# Flip left-right (mirror horizontally)
flipped = tf.image.flip_left_right(image)  

# Show original vs flipped
show_side_by_side(image, flipped, titles=("Original", "Flipped (left-right)"))


ModuleNotFoundError: No module named 'tensorflow.python.data.experimental.ops.iterator_model_ops'